# R&D  Acte d'Engagement : remplissage via GPT-4o Vision

## Pourquoi GPT-4o ?

L'approche classique (mots-clés hardcodés) est fragile : chaque organisme public marocain
utilise sa propre variante du formulaire (labels différents, champs sur la même ligne ou pas,
sections nommées différemment...).

GPT-4o vision **lit et comprend le document visuellement**, quel que soit le format.

## Architecture en 4 étapes

```
┌─────────────────────────────────────────────────────────────┐
│ Étape 1  Sélection des pages (SANS appel API)              │
│   pymupdf lit le texte de chaque page                        │
│   → score par mots-clés (soussigné, CNSS, registre...)      │
│   → on garde les 3 pages les plus pertinentes               │
└──────────────────────┬──────────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────────┐
│ Étape 2  Analyse GPT-4o (1 appel API, pages sélectionnées) │
│   GPT-4o identifie la section "société privée"              │
│   → retourne les bornes Y de la section (y_start%, y_end%)  │
│   → retourne pour chaque champ :                            │
│       • label_text  : texte du libellé (ex: "Numéro CNSS")  │
│       • line_order  : Nème ligne avec points dans la section │
│       • nth         : Nème zone de points sur la ligne      │
│       • x_pct/y_pct : coordonnées (fallback PDFs scannés)   │
└──────────────────────┬──────────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────────┐
│ Étape 3  Remplissage pymupdf (3 niveaux de résolution)     │
│   Niveau 1 (label_text)  : cherche la ligne par son texte   │
│   Niveau 2 (line_order)  : prend la Nème ligne avec points  │
│   Niveau 3 (x_pct/y_pct) : position directe (PDF scanné)   │
└──────────────────────┬──────────────────────────────────────┘
                       ↓
┌─────────────────────────────────────────────────────────────┐
│ Étape 4  Avant / Après + sauvegarde PDF                    │
└─────────────────────────────────────────────────────────────┘
```

**Coût estimé :** ~$0.05–0.10 par document (3 pages × GPT-4o vision)

## Pourquoi 3 informations + 3 niveaux de résolution ?

**Le problème central :** pymupdf ne comprend pas les labels  il travaille uniquement avec des coordonnées (x, y).
GPT-4o doit donc nous donner des indices pour **localiser** la bonne ligne dans le PDF.

---

### Ce que GPT-4o retourne pour chaque champ

GPT-4o regarde l'image et retourne **3 manières de localiser la même ligne**, par ordre de fiabilité décroissante :

```
field_key   : "cnss"
label_text  : "Affiliée à la CNSS sous le numéro"   ← le texte visible avant les points
line_order  : 5                                       ← c'est la 6ème ligne avec des points dans la section
nth         : 0                                       ← 1ère zone de points sur cette ligne
y_pct/x_pct : 52.3 / 18.5                            ← coordonnées en % de page (fallback scanné)
```

---

### Comment pymupdf utilise ces 3 infos (cascade)

```
GPT-4o dit : label="Affiliée à la CNSS", line_order=5, y=52%
                              │
             ┌────────────────┼────────────────┐
             ▼                ▼                ▼
        Niveau 1         Niveau 2         Niveau 3

   pymupdf lit le     pymupdf prend    pymupdf va
   texte du PDF       la 6ème ligne    directement
   et cherche la      qui a des        aux coords
   ligne contenant    pointillés       y=52%, x=18%
   "Affiliée à la     dans la section
   CNSS"

   ✅ PDF natif       ✅ label trop     ✅ PDF scanné
   avec couche        court ("de")      sans texte
   texte              ou absent
```

---

### Pourquoi pas juste les coordonnées y/x ?

GPT-4o est imprécis sur les coordonnées (~5-10% d'erreur).
Sur un A4 (841 pt de hauteur), 5% = **42 points ≈ 1,5 cm** → on tombe sur la mauvaise ligne.

Le texte lui est exact : si pymupdf trouve "Affiliée à la CNSS" dans le PDF, c'est forcément la bonne ligne.

---

> **Résumé :** GPT-4o comprend le document visuellement et donne 3 indices de localisation.
> pymupdf les utilise du plus précis au moins précis pour trouver la bonne ligne et y écrire la valeur.

## Cellule 1  Config & chargement

In [ ]:
import sys, os, base64, json
from pathlib import Path
import fitz
import openai
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image
import io

ROOT   = Path("../..")
INPUT  = Path("input")
OUTPUT = Path("output")
OUTPUT.mkdir(exist_ok=True)

sys.path.insert(0, str(ROOT))

# ── Clé OpenAI ───────────────────────────────────────────────────────────
# Priorité : variable d'environnement → fichier .env du projet
from dotenv import load_dotenv
load_dotenv(ROOT / ".env")
openai.api_key = os.getenv("OPENAI_API_KEY", "")
assert openai.api_key, "❌ OPENAI_API_KEY manquante dans .env"

# ── PDF ──────────────────────────────────────────────────────────────────
pdfs = list(INPUT.glob("*.pdf"))
assert pdfs, "❌ Aucun PDF dans input/"
PDF_PATH = pdfs[0]
print(f"✅ PDF : {PDF_PATH.name}")
print(f"✅ OpenAI key : {openai.api_key[:8]}...")

## Cellule 2  PDF → images base64

On rasterise chaque page en image PNG et on l'encode en base64 pour l'API GPT-4o.
Un seul appel API avec toutes les pages  GPT-4o voit l'intégralité du document.

In [ ]:
def pdf_to_base64_images(pdf_path: Path, dpi: int = 150) -> list[str]:
    """
    Convertit chaque page du PDF en image PNG encodée base64.

    dpi=150 : bon compromis qualité/taille pour GPT-4o vision.
    Au-delà de 200dpi les images deviennent trop lourdes sans gain de précision.

    Retourne : liste de strings base64 (une par page)
    """
    doc = fitz.open(str(pdf_path))
    images_b64 = []
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    for page in doc:
        pix  = page.get_pixmap(matrix=mat, alpha=False)
        png  = pix.tobytes("png")
        images_b64.append(base64.b64encode(png).decode("utf-8"))
    doc.close()
    print(f"✅ {len(images_b64)} page(s) converties en base64 ({dpi} dpi)")
    return images_b64

pages_b64 = pdf_to_base64_images(PDF_PATH)

## Cellule 3  Appel GPT-4o : analyse de la structure

GPT-4o reçoit toutes les pages et doit :
1. Trouver la section **personne morale / société privée** (peu importe comment elle s'appelle)
2. Pour chaque champ à remplir dans cette section : retourner le label, la clé métier et la **position Y en %** depuis le haut de la page

Le JSON retourné sera utilisé à l'étape suivante pour guider pymupdf.

## Cellule 2b  Sélection des pages pertinentes (sans appel API)

Avant d'envoyer quoi que ce soit à GPT-4o, on score chaque page localement via pymupdf
pour identifier les pages qui contiennent une déclaration sur l'honneur / acte d'engagement.

**Logique de scoring :**
- Mots-clés forts (+3) : "déclaration sur l'honneur", "acte d'engagement", "soussigné"
- Mots-clés moyens (+2) : "personne morale", "raison sociale", "capital social"
- Mots-clés faibles (+1) : "cnss", "registre du commerce", "identifiant commun"

On ne garde que les **3 pages avec le score le plus élevé** pour l'appel GPT-4o.

In [ ]:
# Mots-clés avec leur poids  du plus discriminant au plus générique
KEYWORDS_STRONG = [
    "déclaration sur l'honneur", "declaration sur l'honneur",
    "acte d'engagement", "acte dengagement",
    "je soussigné", "je soussignée",
]
KEYWORDS_MEDIUM = [
    "personne morale", "personnes morales",
    "raison sociale", "capital social",
    "cas des sociétés", "société privée",
]
KEYWORDS_WEAK = [
    "cnss", "registre du commerce", "registre de commerce",
    "identifiant commun", "taxe professionnelle",
    "domicile élu", "adresse du siège",
]


def score_page(page: fitz.Page) -> int:
    """
    Score une page selon la densité de mots-clés liés à une déclaration sur l'honneur.
    Travail purement local (pymupdf, pas d'API)  très rapide.
    """
    text = page.get_text("text").lower()
    score = 0
    for kw in KEYWORDS_STRONG:
        if kw in text:
            score += 3
    for kw in KEYWORDS_MEDIUM:
        if kw in text:
            score += 2
    for kw in KEYWORDS_WEAK:
        if kw in text:
            score += 1
    return score


def select_relevant_pages(pdf_path: Path, top_n: int = 3) -> tuple[list[int], list[str]]:
    """
    Score toutes les pages du PDF et retourne les top_n les plus pertinentes.

    Si le PDF a moins de top_n pages, toutes les pages sont retournées.
    Pour les PDFs scannés (score 0 partout), on prend les premières pages par défaut.

    Retourne :
    - page_indices : indices des pages sélectionnées (ordre original du PDF)
    - pages_b64    : images base64 correspondantes (pour l'appel GPT-4o)
    """
    doc = fitz.open(str(pdf_path))
    mat = fitz.Matrix(150 / 72, 150 / 72)

    # Adapter top_n si le PDF a moins de pages
    top_n = min(top_n, doc.page_count)
    print(f"PDF : {doc.page_count} page(s)  sélection des {top_n} plus pertinentes\n")

    scores = []
    for i, page in enumerate(doc):
        s = score_page(page)
        scores.append((i, s))
        bar = '▓' * s if s > 0 else '░ (vide ou scanné)'
        print(f"  Page {i + 1:2d}  score {s:2d}  {bar}")

    # Trier par score décroissant, garder top_n
    scores.sort(key=lambda x: -x[1])
    selected = scores[:top_n]

    # Cas où toutes les pages ont score 0 (PDF scanné sans couche texte)
    if all(s == 0 for _, s in selected):
        print("\n⚠️  Score 0 partout  PDF probablement scanné.")
        print(f"   Sélection des {top_n} premières pages par défaut.")
        selected = [(i, 0) for i in range(top_n)]

    # Remettre dans l'ordre original du document
    selected.sort(key=lambda x: x[0])
    page_indices = [i for i, _ in selected]

    # Générer les images base64 uniquement pour les pages sélectionnées
    pages_b64_selected = []
    for i in page_indices:
        pix = doc[i].get_pixmap(matrix=mat, alpha=False)
        png = pix.tobytes("png")
        pages_b64_selected.append(base64.b64encode(png).decode("utf-8"))

    doc.close()

    print(f"\n✅ Pages envoyées à GPT-4o : {[i + 1 for i in page_indices]}")
    return page_indices, pages_b64_selected


print("── Scoring des pages ────────────────────────────────")
selected_indices, pages_b64_selected = select_relevant_pages(PDF_PATH, top_n=3)

In [ ]:
SYSTEM_PROMPT = """
Tu es un expert en analyse de formulaires administratifs marocains (actes d'engagement, déclarations sur l'honneur).
Tu dois analyser le document et retourner UNIQUEMENT du JSON valide, sans aucun texte autour.
""".strip()

USER_PROMPT = """
Ce document est un formulaire de déclaration sur l'honneur / acte d'engagement marocain.
Tu reçois uniquement les pages pertinentes (sélectionnées au préalable).

Tu dois identifier UNIQUEMENT la sous-section qui concerne une SOCIÉTÉ PRIVÉE COMMERCIALE (SARL, SA, SNC...).

IGNORE ces autres sous-sections :
- "Cas des établissements publics"
- "Cas des coopératives"
- "Cas de l'auto-entrepreneur"
- "Cas des personnes physiques"
- "Cas des groupements"

═══════════════════════════════════════════════════════════
RÈGLES CRITIQUES SUR LES CHAMPS PARTAGEANT LA MÊME LIGNE
═══════════════════════════════════════════════════════════

⚠️ RÈGLE 1  "Je soussigné" et "agissant" sont TOUJOURS sur la même ligne :
   → UN SEUL champ : field_key="signataire_nom", nth=0
   → NE PAS créer qualite_signataire séparément

⚠️ RÈGLE 2  "raison sociale" et "capital social" sont SOUVENT sur la même ligne :
   → raison_sociale : nth=0
   → capital_social : nth=1, MÊME label_text que raison_sociale
   Si sur lignes séparées : nth=0 pour chacun avec son propre label.

⚠️ RÈGLE 3  "rc_localite" et "rc_numero" sont SOUVENT sur la même ligne :
   → rc_localite : nth=0
   → rc_numero   : nth=1, MÊME label_text que rc_localite
   NE PAS utiliser "sous le numéro" seul  trop générique.

⚠️ RÈGLE 4  "telephone" et "fax" peuvent être sur la même ligne :
   → telephone : nth=0,  fax : nth=1, même label_text

═══════════════════════════════════════════════════════════
FORMAT DE CHAQUE CHAMP
═══════════════════════════════════════════════════════════

Pour chaque champ, retourne :
- "field_key"   : clé parmi : signataire_nom, raison_sociale, capital_social,
                  telephone, fax, email, adresse_siege, adresse_domicile,
                  cnss, rc_localite, rc_numero, taxe_pro, ice, rib, autre
- "label_text"  : texte du libellé AVANT les pointillés, SANS points ni hints entre parenthèses.
                  Si le libellé est très court (ex: "de", "au") ou absent → mettre ""
                  ✅ "Numéro téléphone :"    ❌ "Numéro téléphone : ......."
                  ✅ "Je soussigné"          ❌ "Je soussigné:......(nom, prénom)"
                  ✅ ""  (si ligne commence par les points directement)
- "line_order"  : INDEX de cette ligne parmi TOUTES les lignes contenant des zones de points
                  dans la section société, en comptant de haut en bas à partir de 0.
                  Exemple : 0=première ligne avec points, 1=deuxième, etc.
                  ⚠️ COMPTER UNIQUEMENT les lignes qui ont des pointillés à remplir,
                  pas les lignes de texte pur.
- "nth"         : index de la zone de points SUR cette ligne (0=première, 1=deuxième)
- "page"        : index de l'image reçue (0=première image)
- "y_pct"       : position Y %  fallback si PDF scanné
- "x_pct"       : position X % du début des pointillés  fallback si PDF scanné

AUTRES RÈGLES :
- Pour le RIB : nth=1
- Les notes "(7)", "(8)"... NE sont PAS des champs
- NE PAS créer qualite_signataire

Retourne ce JSON et rien d'autre :
{
  "section_label": "Cas des sociétés",
  "y_start_pct": 30.0,
  "y_end_pct": 65.0,
  "page": 0,
  "fields": [
    {
      "field_key": "signataire_nom",
      "label_text": "Je soussigné",
      "line_order": 0,
      "nth": 0,
      "page": 0,
      "y_pct": 35.0,
      "x_pct": 22.0
    },
    {
      "field_key": "raison_sociale",
      "label_text": "",
      "line_order": 1,
      "nth": 0,
      "page": 0,
      "y_pct": 37.0,
      "x_pct": 10.0
    },
    {
      "field_key": "capital_social",
      "label_text": "",
      "line_order": 1,
      "nth": 1,
      "page": 0,
      "y_pct": 37.0,
      "x_pct": 60.0
    }
  ]
}
""".strip()


def analyze_with_gpt4o(pages_b64: list[str], page_indices: list[int]) -> dict:
    """
    Envoie les pages sélectionnées à GPT-4o.
    GPT-4o retourne pour chaque champ :
    - label_text  : libellé textuel (utilisé en priorité pour PDF natif)
    - line_order  : Nème ligne avec points dans la section (fallback si label court/absent)
    - nth         : Nème zone de points sur la ligne
    - y_pct/x_pct : fallback pour PDF scanné
    """
    image_contents = [
        {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64}", "detail": "high"}}
        for b64 in pages_b64
    ]
    image_contents.append({"type": "text", "text": USER_PROMPT})

    client   = openai.OpenAI()
    response = client.chat.completions.create(
        model="gpt-4o",
        messages=[
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user",   "content": image_contents},
        ],
        response_format={"type": "json_object"},
        max_tokens=2000,
        temperature=0,
    )

    raw    = response.choices[0].message.content
    result = json.loads(raw)
    tokens = response.usage.total_tokens

    section_page_rel = result.get("page", 0)
    result["page"] = page_indices[min(section_page_rel, len(page_indices) - 1)]
    for f in result.get("fields", []):
        rel = f.get("page", section_page_rel)
        f["page"] = page_indices[min(rel, len(page_indices) - 1)]

    print(f"✅ GPT-4o  {tokens} tokens ({len(pages_b64)} page(s))")
    print(f"   Section  : {result.get('section_label', '?')}")
    print(f"   Bornes Y : {result.get('y_start_pct', '?')}% → {result.get('y_end_pct', '?')}%")
    print(f"   Page réelle : {result.get('page', 0) + 1}")
    print()
    for f in result.get("fields", []):
        print(f"   {f.get('field_key','?'):20s}  line_order={f.get('line_order','?')}  nth={f.get('nth',0)}  label='{f.get('label_text','?')}'")
    return result


gpt_result = analyze_with_gpt4o(pages_b64_selected, selected_indices)
print("\n── JSON brut ─────────────────────────────────────────")
print(json.dumps(gpt_result, ensure_ascii=False, indent=2))

## Cellule 4  Visualisation des positions GPT-4o

On affiche les positions retournées par GPT-4o sur les pages du document.
Les croix rouges = position Y% GPT-4o → doit tomber sur la bonne ligne.

In [ ]:
def visualize_gpt_positions(pages_b64: list[str], fields: list[dict]):
    """
    Affiche les pages avec les positions retournées par GPT-4o.
    Croix rouge = position (x_pct, y_pct) indiquée par GPT-4o.
    Label = field_key pour identifier visuellement chaque champ.
    """
    # Grouper les champs par page
    by_page: dict[int, list] = {}
    for f in fields:
        by_page.setdefault(f["page"], []).append(f)

    for p_idx, b64 in enumerate(pages_b64):
        png_bytes = base64.b64decode(b64)
        img = Image.open(io.BytesIO(png_bytes))
        w, h = img.size

        fig, ax = plt.subplots(figsize=(8, 11))
        ax.imshow(img)
        ax.axis("off")

        for f in by_page.get(p_idx, []):
            px = f["x_pct"] / 100 * w
            py = f["y_pct"] / 100 * h
            # Croix rouge à la position GPT-4o
            ax.plot(px, py, "r+", markersize=14, markeredgewidth=2)
            # Label field_key en rouge à côté
            ax.text(px + 5, py, f["field_key"], fontsize=7, color="red",
                    va="center", bbox=dict(boxstyle="round,pad=0.2", fc="white", alpha=0.7))

        ax.set_title(f"Page {p_idx + 1}  positions GPT-4o (croix rouges)", fontsize=10)
        plt.tight_layout()
        plt.show()


visualize_gpt_positions(pages_b64, gpt_result.get("fields", []))

## Cellule 5  Remplissage guidé par GPT-4o

Pour chaque champ retourné par GPT-4o :
1. Convertir `y_pct` → position Y absolue sur la page pymupdf
2. Chercher dans rawdict la ligne dont le Y est le plus proche de cette position
3. Dans cette ligne, prendre la `nth` zone de points
4. Effacer + insérer la valeur

C'est pymupdf qui fait le remplissage réel  GPT-4o ne fait qu'indiquer où chercher.

In [ ]:
_DOT_CHARS = set('._…\u2026\u00b7·\u2025\u22ef\u0085')  # tous les caractères "point" utilisés dans les formulaires

# ── Données à insérer ────────────────────────────────────────────────────
# signataire_nom contient NOM + QUALITÉ ensemble car le formulaire a une
# seule zone de points ".....(nom, prénom et qualité)" sur la ligne "Je soussigné"
DATA = {
    "signataire_nom"  : "Mohammed Benali, Directeur Général",
    "raison_sociale"  : "ABI Consulting SARL AU",
    "capital_social"  : "100.000 MAD",
    "telephone"       : "0522 123 456",
    "fax"             : "0522 789 012",
    "email"           : "contact@abi-consulting.ma",
    "adresse_siege"   : "Imm 30, Appt 08, Rue Moulay Ahmed Loukili, Rabat",
    "adresse_domicile": "12 Rue Hassan II, Casablanca",
    "cnss"            : "1234567",
    "rc_localite"     : "Casablanca",
    "rc_numero"       : "RC 145853",
    "taxe_pro"        : "TP 56789012",
    "ice"             : "002579010000023",
    "rib"             : "007780000001234567890123",
}


# ════════════════════════════════════════════════════════════
# FONCTIONS DE DÉTECTION DES ZONES À REMPLIR
# ════════════════════════════════════════════════════════════

def find_dot_zones_in_spans(spans: list) -> list[tuple]:
    """
    Parcourt les caractères d'une ligne (rawdict) et regroupe les séquences
    de points/tirets consécutifs en "zones à remplir".

    Pourquoi rawdict et pas dict ?
    → dict donne la bbox du span entier (ex: "Je soussigné......(nom)")
    → rawdict donne la bbox de CHAQUE caractère → on peut localiser
      exactement où commencent et où finissent les points dans la ligne.

    Extension automatique : si les points sont suivis d'un hint entre
    parenthèses "(nom, prénom et qualité)", on étend le rectangle d'effacement
    jusqu'à la parenthèse fermante pour effacer le hint aussi.

    Retourne : liste de (x0_insert, y0, x1_erase, y1, fontsize)
      x0_insert : x où écrire la valeur (début des points)
      x1_erase  : x jusqu'où effacer (fin des points + hint éventuel)
    """
    zones = []
    for span in spans:
        chars = span.get("chars", [])
        i = 0
        while i < len(chars):
            if chars[i]["c"] not in _DOT_CHARS:
                i += 1
                continue
            # --- début d'une séquence de points ---
            dot_start = i
            while i < len(chars) and chars[i]["c"] in _DOT_CHARS:
                i += 1
            dot_end = i - 1
            if dot_end - dot_start + 1 < 3:  # ignorer les séquences trop courtes
                continue
            x0 = chars[dot_start]["bbox"][0]
            x1 = chars[dot_end]["bbox"][2]
            y0 = min(c["bbox"][1] for c in chars[dot_start:dot_end + 1])
            y1 = max(c["bbox"][3] for c in chars[dot_start:dot_end + 1])
            # --- extension au hint entre parenthèses si présent ---
            erase_x1 = x1
            if i < len(chars) and chars[i]["c"] == '(':
                j = i
                while j < len(chars) and chars[j]["c"] != ')':
                    j += 1
                if j < len(chars):
                    erase_x1 = chars[j]["bbox"][2]
                    i = j + 1
            zones.append((x0, y0, erase_x1, y1, span.get("size", 9.5)))
    return zones


def span_text(span: dict) -> str:
    """Reconstruit le texte d'un span rawdict depuis ses caractères individuels.
    Nécessaire car rawdict ne fournit pas de clé 'text' directement."""
    return "".join(c["c"] for c in span.get("chars", []))


def clean_label(label_text: str) -> str:
    """
    Nettoie le label retourné par GPT-4o avant de l'utiliser comme critère de recherche.

    GPT-4o peut parfois inclure des points ou des hints entre parenthèses dans le label
    malgré les instructions du prompt. Cette fonction les supprime systématiquement.

    Ex: "Je soussigné:......(nom, prénom)" → "Je soussigné:"
    Ex: "Affiliée à la CNSS (7) sous le numéro" → "Affiliée à la CNSS  sous le numéro"
    """
    import re
    cleaned = re.sub(r'[._…\u2026\u00b7·\u2025\u22ef\u0085]{2,}', ' ', label_text)  # supprime les points
    cleaned = re.sub(r'\([^)]*\)', ' ', cleaned)   # supprime les (hints)
    return re.sub(r'\s+', ' ', cleaned).strip()


# ════════════════════════════════════════════════════════════
# FONCTIONS DE LOCALISATION DES LIGNES
# ════════════════════════════════════════════════════════════

def get_all_dot_lines(page: fitz.Page, y_min_abs: float, y_max_abs: float) -> list[tuple]:
    """
    Construit la liste ordonnée de TOUTES les lignes contenant des points à remplir
    dans la section [y_min_abs, y_max_abs].

    Triée de haut en bas → l'index dans cette liste = line_order retourné par GPT-4o.

    Ex pour une section avec 12 champs :
      index 0 → "Je soussigné.....(nom)"
      index 1 → "de.....(raison sociale)....au capital de:....."
      index 2 → "Numéro téléphone : ............."
      ...

    Retourne : liste de (line_y, spans, zones)
    """
    lines_with_dots = []
    for block in page.get_text("rawdict").get("blocks", []):
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            if not spans:
                continue
            line_y = (line["bbox"][1] + line["bbox"][3]) / 2
            if line_y < y_min_abs or line_y > y_max_abs:
                continue
            zones = find_dot_zones_in_spans(spans)
            if zones:
                lines_with_dots.append((line_y, spans, zones))
    lines_with_dots.sort(key=lambda x: x[0])
    return lines_with_dots


def find_line_by_label(page: fitz.Page, label_text: str,
                       y_min_abs: float, y_max_abs: float) -> tuple | None:
    """
    Cherche la ligne contenant le label_text dans la section.

    Algorithme de scoring (du plus strict au plus souple) :
      Score 3 : label nettoyé contenu mot pour mot dans la ligne
      Score 2 : tous les mots-clés significatifs (>3 chars) présents
      Score 1 : premier mot-clé présent

    En cas d'ex-aequo → on prend la ligne la plus haute (première dans le document)
    pour éviter de tomber dans une section adjacente.

    Retourne : (spans, zones) ou None si aucune ligne ne correspond
    """
    label_clean = clean_label(label_text).lower()
    # Ne garder que les mots significatifs (len > 3) pour éviter les faux positifs
    # sur "de", "au", "le"...
    keywords = [w.strip(',:;') for w in label_clean.split() if len(w) > 3][:4]
    if not keywords:
        return None  # label trop court → pas de recherche par texte

    candidates = []
    for block in page.get_text("rawdict").get("blocks", []):
        for line in block.get("lines", []):
            spans = line.get("spans", [])
            if not spans:
                continue
            line_y = (line["bbox"][1] + line["bbox"][3]) / 2
            # Ne regarder que les lignes dans la section société
            if line_y < y_min_abs or line_y > y_max_abs:
                continue
            full  = "".join(span_text(s) for s in spans).lower()
            zones = find_dot_zones_in_spans(spans)
            if not zones:
                continue  # ignorer les lignes sans zone à remplir
            if label_clean in full:
                score = 3
            elif len(keywords) >= 2 and all(kw in full for kw in keywords):
                score = 2
            elif keywords[0] in full:
                score = 1
            else:
                continue
            candidates.append((score, line_y, spans, zones))

    if not candidates:
        return None
    candidates.sort(key=lambda x: (-x[0], x[1]))  # meilleur score, puis plus haut dans page
    _, _, spans, zones = candidates[0]
    return spans, zones


# ════════════════════════════════════════════════════════════
# FONCTIONS DE REMPLISSAGE
# ════════════════════════════════════════════════════════════

def write_value(page: fitz.Page, zones: list, nth: int, value: str) -> None:
    """
    Écrit une valeur dans la nth zone de points d'une ligne.

    Mécanisme :
    1. Rectangle blanc sur la zone → efface les points et le hint éventuel
    2. insert_text au début de la zone → écrit la valeur à la bonne position

    nth_safe : si GPT-4o retourne nth=1 mais qu'il n'y a qu'une seule zone,
    on prend la dernière zone disponible plutôt que de planter.
    """
    nth_safe = min(nth, len(zones) - 1)
    x0, y0, x1_erase, y1, fs = zones[nth_safe]
    # Effacement : rectangle blanc légèrement plus grand que la zone
    page.draw_rect(
        fitz.Rect(x0 - 1, y0 - 1, x1_erase + 1, y1 + 1),
        color=(1, 1, 1), fill=(1, 1, 1),
    )
    # Écriture à la baseline de la zone (y1 - 1 = juste au-dessus du bas)
    page.insert_text(
        fitz.Point(x0, y1 - 1),
        value, fontsize=max(fs, 7.0), color=(0, 0, 0),
    )


def fill_field(page: fitz.Page, label_text: str, line_order: int,
               y_min_abs: float, y_max_abs: float,
               y_target_abs: float, x_target_abs: float,
               nth: int, value: str,
               all_dot_lines: list) -> str:
    """
    Remplit un champ avec 3 niveaux de résolution (priorité décroissante).

    ┌─ Niveau 1 : label_text (PDF natif, label assez long) ─────────────┐
    │  Cherche dans le texte pymupdf la ligne contenant le libellé.      │
    │  Avantage : précis, indépendant de la position dans la section.    │
    │  Limite : échoue si le label est court ("de") ou absent.          │
    └───────────────────────────────────────────────────────────────────┘
           ↓ échec
    ┌─ Niveau 2 : line_order (PDF natif, label court/absent) ───────────┐
    │  Prend la Nème ligne avec des points dans la section.              │
    │  GPT-4o a compté ces lignes dans l'image → index fiable.          │
    │  Avantage : fonctionne même si le label est "de" ou vide.         │
    │  Limite : sensible à l'ordre si deux variantes du formulaire        │
    │           ont des champs dans un ordre différent.                  │
    └───────────────────────────────────────────────────────────────────┘
           ↓ échec (ligne non trouvée = section trop petite ou PDF scanné)
    ┌─ Niveau 3 : coordonnées GPT-4o x_pct/y_pct (PDF scanné) ─────────┐
    │  Pas de couche texte → impossible d'utiliser pymupdf pour lire.   │
    │  On utilise les coordonnées estimées par GPT-4o en % de page.     │
    │  Avantage : fonctionne sur tous types de PDF.                     │
    │  Limite : approximatif (~5-10% d'erreur sur les coordonnées).     │
    └───────────────────────────────────────────────────────────────────┘

    Retourne : "label" | "order" | "scanné" | "echec"
    """
    label_clean = clean_label(label_text)

    # ── Niveau 1 : recherche par texte du libellé ────────────────────────
    if len(label_clean) > 3:  # label trop court → passer directement au niveau 2
        result = find_line_by_label(page, label_clean, y_min_abs, y_max_abs)
        if result is not None:
            _, zones = result
            write_value(page, zones, nth, value)
            return "label"

    # ── Niveau 2 : Nème ligne avec points dans la section ────────────────
    if 0 <= line_order < len(all_dot_lines):
        _, _, zones = all_dot_lines[line_order]
        write_value(page, zones, nth, value)
        return "order"

    # ── Niveau 3 : coordonnées GPT-4o directes (PDF scanné) ─────────────
    if x_target_abs > 0 and y_target_abs > 0:
        line_h = 12  # hauteur estimée d'une ligne en points PDF (≈ 4mm)
        # Rectangle blanc pour masquer les pointillés scannés
        page.draw_rect(
            fitz.Rect(x_target_abs - 1, y_target_abs - line_h,
                      x_target_abs + 300, y_target_abs + 2),
            color=(1, 1, 1), fill=(1, 1, 1),
        )
        page.insert_text(
            fitz.Point(x_target_abs, y_target_abs),
            value, fontsize=9.0, color=(0, 0, 0),
        )
        return "scanné"

    return "echec"


# ════════════════════════════════════════════════════════════
# FONCTION PRINCIPALE DE REMPLISSAGE
# ════════════════════════════════════════════════════════════

def fill_from_gpt_result(pdf_path: Path, gpt_result: dict, data: dict) -> bytes:
    """
    Orchestre le remplissage complet du PDF à partir du résultat GPT-4o.

    Pour chaque champ dans gpt_result["fields"] :
    1. Valide que la page existe et qu'une valeur est disponible dans DATA
    2. Convertit les % de page en coordonnées absolues (points PDF)
    3. Construit (si besoin) le cache des lignes avec points pour la page
    4. Appelle fill_field avec les 3 niveaux de résolution
    5. Affiche le résultat avec le mode utilisé ([label], [order], [scanné])

    Le cache dot_lines_cache évite de relire rawdict pour chaque champ
    d'une même page → gain de performance sur les formulaires multi-champs.
    """
    doc          = fitz.open(str(pdf_path))
    gpt_fields   = gpt_result.get("fields", [])
    section_page = gpt_result.get("page", 0)
    y_start_pct  = gpt_result.get("y_start_pct", 0)
    y_end_pct    = gpt_result.get("y_end_pct", 100)

    # ── Détection préventive des doublons ────────────────────────────────
    # Un doublon = deux champs qui ciblent le même (label, line_order, nth, page)
    # → le second champ écrase le premier silencieusement sans ce check
    seen = {}
    for f in gpt_fields:
        key = (f.get("label_text", ""), f.get("line_order", -1), f.get("nth", 0), f.get("page", 0))
        if key in seen:
            print(f"  ⚠️  DOUBLON : line_order={f.get('line_order')} nth={f.get('nth')} "
                  f"→ '{seen[key]}' et '{f.get('field_key')}' ciblent la même zone !")
        seen[key] = f.get("field_key", "")

    filled, skipped = 0, 0
    dot_lines_cache: dict[int, list] = {}  # cache par numéro de page

    for f in gpt_fields:
        field_key  = f.get("field_key", "")
        label_text = f.get("label_text", "")
        line_order = f.get("line_order", -1)
        value      = data.get(field_key, "")
        page_idx   = f.get("page", section_page)
        y_pct      = f.get("y_pct", 50)
        x_pct      = f.get("x_pct", 0)
        nth        = f.get("nth", 0)

        if not value:
            print(f"  ⚠️  {field_key:20s}  pas de valeur dans DATA")
            skipped += 1
            continue
        if page_idx >= len(doc):
            print(f"  ⚠️  {field_key:20s}  page {page_idx} inexistante dans le PDF")
            skipped += 1
            continue

        page      = doc[page_idx]
        page_h    = page.rect.height   # hauteur de la page en points (A4 ≈ 841pt)
        page_w    = page.rect.width    # largeur de la page en points (A4 ≈ 595pt)

        # Conversion % → points absolus pour les bornes de section et le fallback scanné
        y_min_abs = y_start_pct / 100 * page_h
        y_max_abs = y_end_pct   / 100 * page_h
        y_abs     = y_pct / 100 * page_h   # utilisé uniquement si PDF scanné
        x_abs     = x_pct / 100 * page_w   # utilisé uniquement si PDF scanné

        # Construction du cache pour cette page (une seule fois par page)
        if page_idx not in dot_lines_cache:
            dot_lines_cache[page_idx] = get_all_dot_lines(page, y_min_abs, y_max_abs)
        all_dot_lines = dot_lines_cache[page_idx]

        mode = fill_field(page, label_text, line_order,
                          y_min_abs, y_max_abs, y_abs, x_abs,
                          nth, value, all_dot_lines)

        label_display = clean_label(label_text)[:18] or f"line#{line_order}"
        if mode != "echec":
            print(f"  ✅ {field_key:20s} [{mode:6s}] nth={nth}  '{label_display}' → '{value[:28]}'")
            filled += 1
        else:
            print(f"  ❌ {field_key:20s}  '{label_display}' / line#{line_order} non trouvé")
            skipped += 1

    print(f"\nRésultat : {filled} champs remplis, {skipped} ignorés")
    out = io.BytesIO()
    doc.save(out)
    doc.close()
    return out.getvalue()


# ── Lancement du remplissage ─────────────────────────────────────────────
result_bytes = fill_from_gpt_result(PDF_PATH, gpt_result, DATA)

## Cellule 6  Avant / Après + Sauvegarde

In [ ]:
def page_to_img(page, dpi=120):
    mat = fitz.Matrix(dpi / 72, dpi / 72)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    return Image.frombytes("RGB", [pix.width, pix.height], pix.samples)


doc_orig   = fitz.open(str(PDF_PATH))
doc_result = fitz.open(stream=result_bytes, filetype="pdf")

for p_idx in range(doc_orig.page_count):
    fig, axes = plt.subplots(1, 2, figsize=(16, 11))
    axes[0].imshow(page_to_img(doc_orig[p_idx]))
    axes[0].set_title(f"AVANT  Page {p_idx + 1}", fontsize=10)
    axes[0].axis("off")
    axes[1].imshow(page_to_img(doc_result[p_idx]))
    axes[1].set_title(f"APRÈS (GPT-4o)  Page {p_idx + 1}", fontsize=10)
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()

# Sauvegarde
out_path = OUTPUT / "acte_engagement_gpt4o.pdf"
out_path.write_bytes(result_bytes)
print(f"✅ Sauvegardé → {out_path}")